# Rap Songs Filter

This notebook filters `data/song_lyrics.csv` down to rows where `tag` is `rap` using Polars lazy scanning. The source CSV is large, so the workflow avoids loading the full file into memory and writes a reusable Parquet cache for faster follow-up filtering.

In [1]:
from pathlib import Path
import polars as pl

DATA_DIR = Path("data")
CSV_PATH = DATA_DIR / "song_lyrics.csv"

CSV_PATH, CSV_PATH.exists(), CSV_PATH.stat().st_size / 1_000_000_000

(WindowsPath('data/song_lyrics.csv'), True, 9.070394868)

## Configure Columns

Keep `lyrics` if you need full text analysis. Set `INCLUDE_LYRICS = False` if you mostly want metadata; it will reduce memory use and make the filtered dataframe much smaller.

In [2]:
INCLUDE_LYRICS = True
REBUILD_CACHE = False

cache_suffix = "with_lyrics" if INCLUDE_LYRICS else "metadata_only"
RAP_PARQUET_PATH = DATA_DIR / f"rap_songs_{cache_suffix}.parquet"

base_columns = [
    "title",
    "tag",
    "artist",
    "year",
    "views",
    "features",
    "id",
    "language_cld3",
    "language_ft",
    "language",
]
keep_columns = base_columns + (["lyrics"] if INCLUDE_LYRICS else [])

schema_overrides = {
    "title": pl.Utf8,
    "tag": pl.Utf8,
    "artist": pl.Utf8,
    "year": pl.Int32,
    "views": pl.Int64,
    "features": pl.Utf8,
    "lyrics": pl.Utf8,
    "id": pl.Int64,
    "language_cld3": pl.Utf8,
    "language_ft": pl.Utf8,
    "language": pl.Utf8,
}

RAP_PARQUET_PATH, keep_columns

(WindowsPath('data/rap_songs_with_lyrics.parquet'),
 ['title',
  'tag',
  'artist',
  'year',
  'views',
  'features',
  'id',
  'language_cld3',
  'language_ft',
  'language',
  'lyrics'])

## Build a Lazy Rap Filter

`scan_csv` creates a lazy query plan. Filtering and column selection are applied while Polars streams through the file instead of materializing the entire 9 GB CSV first.

In [3]:
songs_lf = pl.scan_csv(
    CSV_PATH,
    schema_overrides=schema_overrides,
    infer_schema_length=1_000,
    null_values=["", "NA", "N/A", "null", "None"],
)

available_columns = songs_lf.collect_schema().names()
selected_columns = [column for column in keep_columns if column in available_columns]

rap_lf = (
    songs_lf
    .filter(pl.col("tag").str.to_lowercase() == "rap")
    .select(selected_columns)
)

rap_lf

## Preview Without Loading Everything

In [4]:
rap_lf.head(20).collect()

title,tag,artist,year,views,features,id,language_cld3,language_ft,language,lyrics
str,str,str,i32,i64,str,i64,str,str,str,str
"""Killa Cam""","""rap""","""Cam'ron""",2004,173166,"""{""Cam\\'ron"",""Opera Steve""}""",1,"""en""","""en""","""en""","""[Chorus: Opera Steve & Cam'ron…"
"""Can I Live""","""rap""","""JAY-Z""",1996,468624,"""{}""",3,"""en""","""en""","""en""","""[Produced by Irv Gotti] [Intr…"
"""Forgive Me Father""","""rap""","""Fabolous""",2003,4743,"""{}""",4,"""en""","""en""","""en""","""Maybe cause I'm eatin And thes…"
"""Down and Out""","""rap""","""Cam'ron""",2004,144404,"""{""Cam\\'ron"",""Kanye West"",""Syl…",5,"""en""","""en""","""en""","""[Produced by Kanye West and Br…"
"""Fly In""","""rap""","""Lil Wayne""",2005,78271,"""{}""",6,"""en""","""en""","""en""","""[Intro] So they ask me ""Young …"
…,…,…,…,…,…,…,…,…,…,…
"""What Happened to That Boy?""","""rap""","""Birdman""",2002,100347,"""{Clipse}""",17,"""en""","""en""","""en""","""[Intro: Birdman] Ayy-ayy, ayy,…"
"""Its Hot Some Like It Hot""","""rap""","""JAY-Z""",1999,103549,"""{}""",18,"""en""","""en""","""en""","""[Produced by Timbaland] [Vers…"
"""Losing Weight Pt. 2""","""rap""","""Cam'ron""",2002,32712,"""{""Cam\\'ron"",""Juelz Santana""}""",19,"""en""","""en""","""en""","""[Chorus: Cam'ron] Ayo, fuck lo…"


## Write a Fast Parquet Cache

Run this once. Future notebook sessions can load `data/rap_songs.parquet`, which should be much faster and smaller than reparsing the full CSV.

In [5]:
if REBUILD_CACHE and RAP_PARQUET_PATH.exists():
    RAP_PARQUET_PATH.unlink()

if not RAP_PARQUET_PATH.exists():
    rap_lf.sink_parquet(RAP_PARQUET_PATH, compression="zstd")

RAP_PARQUET_PATH, RAP_PARQUET_PATH.exists(), RAP_PARQUET_PATH.stat().st_size / 1_000_000

(WindowsPath('data/rap_songs_with_lyrics.parquet'), True, 1553.487242)

## Load the Rap Songs Dataframe

`rap_df` is an in-memory Polars dataframe for interactive filtering. If this is still too large with lyrics included, set `INCLUDE_LYRICS = False`, rerun the notebook from the top, and rebuild the Parquet cache.

In [6]:
rap_df = pl.read_parquet(RAP_PARQUET_PATH)
rap_df.shape, rap_df.head()

((1724816, 11),
 shape: (5, 11)
 ┌──────────────┬─────┬───────────┬──────┬───┬──────────────┬─────────────┬──────────┬──────────────┐
 │ title        ┆ tag ┆ artist    ┆ year ┆ … ┆ language_cld ┆ language_ft ┆ language ┆ lyrics       │
 │ ---          ┆ --- ┆ ---       ┆ ---  ┆   ┆ 3            ┆ ---         ┆ ---      ┆ ---          │
 │ str          ┆ str ┆ str       ┆ i32  ┆   ┆ ---          ┆ str         ┆ str      ┆ str          │
 │              ┆     ┆           ┆      ┆   ┆ str          ┆             ┆          ┆              │
 ╞══════════════╪═════╪═══════════╪══════╪═══╪══════════════╪═════════════╪══════════╪══════════════╡
 │ Killa Cam    ┆ rap ┆ Cam'ron   ┆ 2004 ┆ … ┆ en           ┆ en          ┆ en       ┆ [Chorus:     │
 │              ┆     ┆           ┆      ┆   ┆              ┆             ┆          ┆ Opera Steve  │
 │              ┆     ┆           ┆      ┆   ┆              ┆             ┆          ┆ & Cam'ron…   │
 │ Can I Live   ┆ rap ┆ JAY-Z     ┆ 1996 ┆ … ┆ en 

In [17]:
import polars as pl
 
artist_category_map = {
    # Metadata / non-artist pages
    "Genius English Translations": "Other / Non-rap / Metadata",
    "Genius Brasil Tradues": "Other / Non-rap / Metadata",
    "Genius Brasil Traducoes": "Other / Non-rap / Metadata",
    "Genius Traducciones al Espaol": "Other / Non-rap / Metadata",
    "Genius Traducciones al Espanol": "Other / Non-rap / Metadata",
    "Genius Russian Translations": "Other / Non-rap / Metadata",
    "Genius Traductions Franaises": "Other / Non-rap / Metadata",
    "Genius Traductions Francaises": "Other / Non-rap / Metadata",
    "Genius Trke eviri": "Other / Non-rap / Metadata",
    "Genius Turkce Ceviri": "Other / Non-rap / Metadata",
    "Genius Romanizations": "Other / Non-rap / Metadata",
    "Polskie tumaczenia Genius": "Other / Non-rap / Metadata",
    "Genius Deutsche bersetzungen": "Other / Non-rap / Metadata",
    "Genius Nederlandse Vertalingen": "Other / Non-rap / Metadata",
    "Genius Farsi Translations": "Other / Non-rap / Metadata",
    "Genius Arabic Translations": "Other / Non-rap / Metadata",
    "Spotify": "Other / Non-rap / Metadata",
    "Unknown Artist": "Other / Non-rap / Unknown",
    "Various Artists": "Other / Non-rap / Compilation",

    # Major mainstream / pop rap
    "Drake": "Mainstream / Pop Rap",
    "Kanye West": "Mainstream / Pop Rap",
    "Lil Wayne": "Mainstream / Pop Rap",
    "Nicki Minaj": "Mainstream / Pop Rap",
    "Wiz Khalifa": "Mainstream / Pop Rap",
    "Big Sean": "Mainstream / Pop Rap",
    "G-Eazy": "Mainstream / Pop Rap",
    "B.o.B": "Mainstream / Pop Rap",
    "T-Pain": "Mainstream / Pop Rap",
    "Tyga": "Mainstream / Pop Rap",
    "Post Malone": "Mainstream / Pop Rap",
    "Pitbull": "Pop Rap",
    "Nelly": "Pop Rap",
    "Flo Rida": "Pop Rap",
    "Kid Ink": "Mainstream / Pop Rap",
    "Chris Brown": "R&B / Pop / Rap-adjacent",
    "Akon": "R&B / Pop / Rap-adjacent",
    "Frank Ocean": "R&B / Pop / Rap-adjacent",
    "Trey Songz": "R&B / Pop / Rap-adjacent",

    # Trap / Southern / street
    "Gucci Mane": "Trap / Southern Rap",
    "Future": "Trap / Southern Rap",
    "Young Thug": "Trap / Southern Rap",
    "Migos": "Trap / Southern Rap",
    "2 Chainz": "Trap / Southern Rap",
    "Waka Flocka Flame": "Trap / Southern Rap",
    "Jeezy": "Trap / Southern Rap",
    "T.I.": "Trap / Southern Rap",
    "Yo Gotti": "Trap / Southern Rap",
    "Rick Ross": "Trap / Southern Rap",
    "Juicy J": "Trap / Southern Rap",
    "Project Pat": "Trap / Southern Rap",
    "Three 6 Mafia": "Trap / Southern Rap",
    "Ace Hood": "Trap / Southern Rap",
    "DaBaby": "Trap / Southern Rap",
    "Money Man": "Trap / Southern Rap",
    "Moneybagg Yo": "Trap / Southern Rap",
    "Young Dolph": "Trap / Southern Rap",
    "Rich The Kid": "Trap / Southern Rap",
    "Lil Baby": "Trap / Southern Rap",
    "Gunna": "Trap / Southern Rap",
    "21 Savage": "Trap / Southern Rap",
    "Offset": "Trap / Southern Rap",
    "Quavo": "Trap / Southern Rap",
    "Peewee Longway": "Trap / Southern Rap",
    "OJ da Juiceman": "Trap / Southern Rap",
    "Young Scooter": "Trap / Southern Rap",
    "Hoodrich Pablo Juan": "Trap / Southern Rap",

    # Drill
    "Chief Keef": "Drill",
    "Lil Durk": "Drill",
    "G Herbo": "Drill",
    "King Von": "Drill",
    "Fredo Santana": "Drill",
    "Lil Reese": "Drill",
    "FBG Duck": "Drill",
    "Pop Smoke": "Drill",
    "Fivio Foreign": "Drill",
    "Tee Grizzley": "Street Rap / Drill-adjacent",

    # Street / gangsta
    "50 Cent": "Gangsta / Street Rap",
    "The Game": "Gangsta / Street Rap",
    "DMX": "Gangsta / Street Rap",
    "Meek Mill": "Street Rap",
    "Lloyd Banks": "Gangsta / Street Rap",
    "Styles P": "Gangsta / Street Rap",
    "Jadakiss": "Gangsta / Street Rap",
    "Pusha T": "Gangsta / Street Rap",
    "Kevin Gates": "Street Rap",
    "Kodak Black": "Street Rap",
    "Boosie Badazz": "Street Rap",
    "YoungBoy Never Broke Again": "Street Rap",
    "YG": "West Coast / Street Rap",
    "Nipsey Hussle": "West Coast / Street Rap",
    "Dave East": "East Coast / Street Rap",
    "Mozzy": "West Coast / Street Rap",
    "Benny the Butcher": "Street Rap / Boom Bap",
    "Westside Gunn": "Street Rap / Boom Bap",
    "Conway the Machine": "Street Rap / Boom Bap",
    "Freddie Gibbs": "Street Rap / Lyrical Rap",

    # Emo / melodic / SoundCloud
    "Juice WRLD": "Emo Rap / Melodic Rap",
    "Lil Uzi Vert": "Emo Rap / Melodic Rap",
    "Trippie Redd": "Emo Rap / Melodic Rap",
    "XXXTENTACION": "Emo Rap / Melodic Rap",
    "Lil Peep": "Emo Rap / Melodic Rap",
    "Lil Tracy": "Emo Rap / Melodic Rap",
    "The Kid LAROI": "Emo Rap / Melodic Rap",
    "iann dior": "Emo Rap / Melodic Rap",
    "Machine Gun Kelly": "Emo Rap / Pop Rap",
    "A Boogie wit da Hoodie": "Melodic Rap",
    "Rod Wave": "Melodic Rap",
    "Don Toliver": "Melodic Rap",
    "PnB Rock": "Melodic Rap",
    "Tory Lanez": "Melodic Rap",
    "Fetty Wap": "Melodic Rap",
    "Lil Tjay": "Melodic Rap",
    "Toosii": "Melodic Rap",
    "Polo G": "Melodic Rap / Street Rap",

    # SoundCloud / rage / plugg
    "Playboi Carti": "Rage / Plugg / SoundCloud",
    "Yeat": "Rage / Plugg / SoundCloud",
    "Summrs": "Rage / Plugg / SoundCloud",
    "Destroy Lonely": "Rage / Plugg / SoundCloud",
    "Ken Car$on": "Rage / Plugg / SoundCloud",
    "Ken Carson": "Rage / Plugg / SoundCloud",
    "Lucki": "Cloud Rap / Plugg",
    "Kankan": "Plugg / Underground",
    "UnoTheActivist": "SoundCloud Rap",
    "Thouxanbanfauni": "SoundCloud Rap",
    "Famous Dex": "SoundCloud Rap",
    "Lil Yachty": "SoundCloud Rap / Pop Rap",
    "Lil Pump": "SoundCloud Rap",
    "Smokepurpp": "SoundCloud Rap",
    "Ski Mask the Slump God": "SoundCloud Rap",
    "Comethazine": "SoundCloud Rap",
    "SoFaygo": "Rage / SoundCloud Rap",
    "Duwap Kaine": "Plugg / Underground",
    "iayze": "Plugg / Underground",

    # Dark trap / horrorcore / trap metal
    "BONES": "Dark Trap / Underground Rap",
    "Bones": "Dark Trap / Underground Rap",
    "$UICIDEBOY$": "Dark Trap / Underground Rap",
    "Ghostemane": "Dark Trap / Industrial Rap",
    "Scarlxrd": "Trap Metal",
    "City Morgue": "Trap Metal",
    "ZillaKami": "Trap Metal",
    "SosMula": "Trap Metal",
    "Insane Clown Posse": "Horrorcore / Dark Rap",
    "Twiztid": "Horrorcore / Dark Rap",
    "Brotha Lynch Hung": "Horrorcore / Dark Rap",
    "Necro": "Horrorcore / Dark Rap",
    "Esham": "Horrorcore / Dark Rap",
    "Twisted Insane": "Horrorcore / Fast Rap",

    # Lyrical / conscious / technical
    "Kendrick Lamar": "Lyrical / Conscious Rap",
    "J. Cole": "Lyrical / Conscious Rap",
    "Lupe Fiasco": "Lyrical / Conscious Rap",
    "Logic": "Lyrical / Conscious Rap",
    "Big K.R.I.T.": "Lyrical / Conscious Rap",
    "Wale": "Lyrical / Conscious Rap",
    "Common": "Lyrical / Conscious Rap",
    "Talib Kweli": "Lyrical / Conscious Rap",
    "The Roots": "Lyrical / Conscious Rap",
    "Ab-Soul": "Lyrical / Conscious Rap",
    "Jay Electronica": "Lyrical / Conscious Rap",
    "Rapsody": "Lyrical / Conscious Rap",
    "Little Simz": "Lyrical / Conscious Rap",
    "JID": "Lyrical / Technical Rap",
    "Denzel Curry": "Lyrical / Aggressive Rap",
    "Vince Staples": "Alternative / Lyrical Rap",
    "Earl Sweatshirt": "Alternative / Lyrical Rap",
    "Mick Jenkins": "Lyrical / Conscious Rap",
    "Joey Bada$$": "Lyrical / Boom Bap",
    "Cordae": "Lyrical / Conscious Rap",
    "Joyner Lucas": "Technical / Lyrical Rap",
    "Hopsin": "Technical / Lyrical Rap",
    "K.A.A.N.": "Technical / Lyrical Rap",
    "Eminem": "Technical / Lyrical Rap",
    "Royce da 5'9\"": "Technical / Lyrical Rap",
    "Busta Rhymes": "Technical / Classic Rap",
    "Twista": "Technical / Fast Rap",
    "Tech N9ne": "Technical / Independent Rap",

    # Classic / old school / golden age
    "2Pac": "Classic Hip-Hop",
    "The Notorious B.I.G.": "Classic Hip-Hop",
    "Nas": "Classic Hip-Hop",
    "JAY-Z": "Classic Hip-Hop",
    "Snoop Dogg": "Classic Hip-Hop",
    "Ice Cube": "Classic Hip-Hop",
    "Dr. Dre": "Classic Hip-Hop",
    "E-40": "Bay Area / Classic Rap",
    "Too $hort": "Bay Area / Classic Rap",
    "Mac Dre": "Bay Area / Classic Rap",
    "Master P": "Southern Classic Rap",
    "UGK": "Southern Classic Rap",
    "Scarface": "Southern Classic Rap",
    "Geto Boys": "Southern Classic Rap",
    "OutKast": "Southern Classic Rap",
    "Ludacris": "Southern Classic Rap",
    "Chamillionaire": "Southern Rap",
    "Z-Ro": "Southern Rap",
    "Bun B": "Southern Classic Rap",
    "Wu-Tang Clan": "Classic Hip-Hop",
    "Raekwon": "Classic Hip-Hop",
    "Ghostface Killah": "Classic Hip-Hop",
    "Method Man": "Classic Hip-Hop",
    "Redman": "Classic Hip-Hop",
    "RZA": "Classic Hip-Hop",
    "Inspectah Deck": "Classic Hip-Hop",
    "Ol' Dirty Bastard": "Classic Hip-Hop",
    "KRS-One": "Classic Hip-Hop",
    "LL Cool J": "Classic Hip-Hop",
    "Public Enemy": "Classic Hip-Hop",
    "A Tribe Called Quest": "Classic Hip-Hop",
    "De La Soul": "Classic Hip-Hop",
    "Big L": "Classic Hip-Hop",
    "Mobb Deep": "Classic Hip-Hop",
    "Gang Starr": "Classic Hip-Hop",
    "Big Daddy Kane": "Classic Hip-Hop",
    "EPMD": "Classic Hip-Hop",
    "Naughty By Nature": "Classic Hip-Hop",
    "Cypress Hill": "Classic Hip-Hop",
    "Bone Thugs-N-Harmony": "Classic Hip-Hop",

    # Underground / abstract / experimental
    "MF DOOM": "Underground / Abstract Rap",
    "Kool Keith": "Underground / Abstract Rap",
    "Aesop Rock": "Abstract / Experimental Rap",
    "billy woods": "Abstract / Experimental Rap",
    "JPEGMAFIA": "Experimental Rap",
    "Death Grips": "Experimental Rap",
    "Run The Jewels": "Alternative Rap",
    "Danny Brown": "Alternative / Experimental Rap",
    "Open Mike Eagle": "Alternative / Abstract Rap",
    "Busdriver": "Alternative / Abstract Rap",
    "Milo": "Alternative / Abstract Rap",
    "milo": "Alternative / Abstract Rap",
    "Blu": "Underground / Alternative Rap",
    "Murs": "Underground / Alternative Rap",
    "Atmosphere": "Underground / Independent Rap",
    "Brother Ali": "Underground / Independent Rap",
    "Immortal Technique": "Underground / Political Rap",
    "Canibus": "Underground / Lyrical Rap",
    "K-Rino": "Underground / Lyrical Rap",
    "Killah Priest": "Underground / Lyrical Rap",
    "R.A. The Rugged Man": "Underground / Lyrical Rap",
    "Vinnie Paz": "Underground / Hardcore Rap",
    "Jedi Mind Tricks": "Underground / Hardcore Rap",
    "Apathy": "Underground / Lyrical Rap",
    "Roc Marciano": "Underground / Boom Bap",
    "Sean Price": "Underground / Boom Bap",
    "Tha God Fahim": "Underground / Boom Bap",

    # Battle rap / platforms
    "URLtv": "Battle Rap / Platform",
    "King of the Dot": "Battle Rap / Platform",
    "Rap Contenders": "Battle Rap / Platform",
    "Versus Battle": "Battle Rap / Platform",
    "Don't Flop": "Battle Rap / Platform",
    "Tsu Surf": "Battle Rap",
    "Loaded Lux": "Battle Rap",
    "Daylyt": "Battle Rap",

    # International rap buckets
    "Skepta": "UK Rap / Grime",
    "Wiley": "UK Rap / Grime",
    "Stormzy": "UK Rap / Grime",
    "Giggs": "UK Rap",
    "AJ Tracey": "UK Rap / Grime",
    "Kano": "UK Rap / Grime",
    "Dizzee Rascal": "UK Rap / Grime",
    "Headie One": "UK Drill",
    "Nines": "UK Rap",
    "D-Block Europe": "UK Rap",
    "Tinie Tempah": "UK Rap / Pop Rap",
}

In [23]:
artist_category_map.update({
    # Internet / cloud / underground / oddball
    "Lil B": "Cloud Rap / Internet Rap",
    "Soulja Boy": "Internet Rap / Early Trap",
    "Charles Hamilton": "Blog Era / Lyrical Rap",
    "Sybyr": "Experimental / Dark Rap",
    "Chris Travis": "Cloud Rap / Underground Rap",
    "Xavier Wulf": "Cloud Rap / Underground Rap",
    "Sickboyrari": "Cloud Rap / Underground Rap",
    "Yung Lean": "Cloud Rap",
    "RiFF RAFF": "Internet Rap / Pop Rap",
    "Viper": "Outsider / Internet Rap",
    "Rozz Dyliams": "Dark Underground Rap",
    "OmenXIII": "Dark Trap / Underground Rap",
    "Lil Ugly Mane": "Experimental / Underground Rap",

    # Southern / trap / street
    "Curren$y": "Southern / Stoner Rap",
    "French Montana": "Mainstream / Street Rap",
    "Plies": "Southern / Street Rap",
    "NLE Choppa": "Street Rap / Drill-adjacent",
    "Travis Scott": "Trap / Psychedelic Rap",
    "Ace Hood": "Southern / Street Rap",
    "Fabolous": "East Coast / Mainstream Rap",
    "Joe Budden": "East Coast / Lyrical Rap",
    "Trae tha Truth": "Southern / Street Rap",
    "Lil Tecca": "Melodic Rap",
    "DaBaby": "Trap / Southern Rap",
    "Project Pat": "Southern Classic Rap",
    "Juicy J": "Southern Classic Rap",
    "Three 6 Mafia": "Southern Classic Rap",
    "Master P": "Southern Classic Rap",
    "Z-Ro": "Southern Rap",
    "Chamillionaire": "Southern Rap",
    "Too $hort": "Bay Area / Classic Rap",
    "Mac Dre": "Bay Area / Classic Rap",
    "E-40": "Bay Area / Classic Rap",

    # Melodic / emo / pop-adjacent
    "Mac Miller": "Alternative / Melodic Rap",
    "Russ": "Pop Rap / Independent Rap",
    "Tory Lanez": "Melodic Rap",
    "Fetty Wap": "Melodic Rap",
    "TheHxliday": "Melodic Rap",
    "Lil Mosey": "Melodic Rap",
    "Don Toliver": "Melodic Rap",
    "Sewerperson": "Emo Rap / Underground Rap",
    "Cold Hart": "Emo Rap / Underground Rap",
    "Oliver Francis": "Cloud Rap / Melodic Rap",

    # Lyrical / technical / underground
    "K.A.A.N.": "Technical / Lyrical Rap",
    "Kool Savas": "Technical / German Rap",
    "K-Rino": "Underground / Lyrical Rap",
    "Killah Priest": "Underground / Lyrical Rap",
    "Papoose": "East Coast / Lyrical Rap",
    "Crooked I": "Technical / Lyrical Rap",
    "SkyBlew": "Conscious / Underground Rap",
    "Napoleon Da Legend": "Underground / Lyrical Rap",
    "Horseshoe G.A.N.G": "Technical / Lyrical Rap",
    "Tha God Fahim": "Underground / Boom Bap",
    "Roc Marciano": "Underground / Boom Bap",
    "Westside Gunn": "Street Rap / Boom Bap",
    "Conway the Machine": "Street Rap / Boom Bap",
    "Freddie Gibbs": "Street Rap / Lyrical Rap",

    # Classic / legacy
    "Busta Rhymes": "Technical / Classic Rap",
    "Royce da 5'9\"": "Technical / Lyrical Rap",
    "Mobb Deep": "Classic Hip-Hop",
    "Redman": "Classic Hip-Hop",
    "Scarface": "Southern Classic Rap",
    "Bone Thugs-N-Harmony": "Classic Hip-Hop",
    "Ludacris": "Southern Classic Rap",
    "Cam'ron": "East Coast / Classic Rap",
    "Twista": "Technical / Fast Rap",
    "Daz Dillinger": "West Coast / Classic Rap",
    "MC Eiht": "West Coast / Classic Rap",

    # Drill / street variants
    "Lil Reese": "Drill",
    "FBG Duck": "Drill",
    "Lil Durk": "Drill",
    "Chief Keef": "Drill",
    "G Herbo": "Drill",
    "King Von": "Drill",
    "Polo G": "Melodic Rap / Street Rap",
    "Tee Grizzley": "Street Rap / Drill-adjacent",

    # International rap
    "JuL": "French Rap",
    "Booba": "French Rap",
    "Rohff": "French Rap",
    "La Fouine": "French Rap",
    "Vald": "French Rap",
    "Ninho": "French Rap",
    "Lacrim": "French Rap",
    "Niro": "French Rap",
    "Alkpote": "French Rap",
    "Disiz": "French Rap",
    "IAM": "French Rap",
    "Guizmo": "French Rap",
    "Kery James": "French Rap",
    "MC Solaar": "French Rap",
    "Orelsan": "French Rap",
    "Nekfeu": "French Rap",
    "Soprano": "French Rap",

    "Kollegah": "German Rap",
    "Bushido": "German Rap",
    "Fler": "German Rap",
    "Eko Fresh": "German Rap",
    "Money Boy": "German Rap",
    "Prinz Pi": "German Rap",
    "Samy Deluxe": "German Rap",
    "Farid Bang": "German Rap",
    "Sido": "German Rap",
    "Kontra K": "German Rap",
    "Massiv": "German Rap",
    "Haftbefehl": "German Rap",
    "Bonez MC": "German Rap",
    "Capital Bra": "German Rap",
    "Ufo361": "German Rap",

    "Tede": "Polish Rap",
    "O.S.T.R.": "Polish Rap",
    "Pikers": "Polish Rap",
    "Paluch": "Polish Rap",
    "Quebonafide": "Polish Rap",
    "Taco Hemingway": "Polish Rap",

    "Vybz Kartel": "Dancehall / Rap-adjacent",
    "Popcaan": "Dancehall / Rap-adjacent",
    "Alkaline": "Dancehall / Rap-adjacent",
    "Daddy Yankee": "Reggaeton / Rap-adjacent",
    "Anuel AA": "Latin Trap / Reggaeton",
    "Duki": "Latin Trap / Rap",
    "C. Tangana": "Spanish Rap / Latin Rap",
    "Cartel de Santa": "Mexican Rap",
    "Gera MX": "Mexican Rap",

    # YouTube / nerdcore / meme / nontraditional rap
    "Dan Bull": "Nerdcore / YouTube Rap",
    "JT Music": "Nerdcore / YouTube Rap",
    "Rustage": "Nerdcore / YouTube Rap",
    "Shwabadi": "Nerdcore / YouTube Rap",
    "None Like Joshua": "Nerdcore / YouTube Rap",
    "Daddyphatsnaps": "Nerdcore / YouTube Rap",
    "GameboyJones": "Nerdcore / YouTube Rap",
    "Fabvl": "Nerdcore / YouTube Rap",
    "Akira The Don": "Spoken Word / Rap-adjacent",
    "The Lonely Island": "Comedy Rap",
    "Epic Rap Battles of History": "Comedy / Battle Rap",
    "Lin-Manuel Miranda": "Musical Theater / Rap-adjacent",

    # Likely non-rap / noise / uncertain metadata
    "Mario William Vitale": "Other / Non-rap / Uncategorized",
    "Muze Sikk": "Other / Non-rap / Uncategorized",
    "OCTOBERSFULLMOON": "Other / Non-rap / Uncategorized",
    "Wild Wes": "Other / Non-rap / Uncategorized",
    "888 Blue": "Other / Non-rap / Uncategorized",
    "Zoan": "Other / Non-rap / Uncategorized",
    "Yamine": "Other / Non-rap / Uncategorized",
    "j. sula": "Other / Non-rap / Uncategorized",
})

In [29]:
artist_category_map.update({
    # Keep noisy / uncertain high-volume accounts in other
    "Muze Sikk": "Other / Non-rap / Uncategorized",
    "OCTOBERSFULLMOON": "Other / Non-rap / Uncategorized",
    "Mario William Vitale": "Other / Non-rap / Uncategorized",
    "Wild Wes": "Other / Non-rap / Uncategorized",
    "Yamine": "Other / Non-rap / Uncategorized",
    "T.W.R.": "Other / Non-rap / Uncategorized",
    "Zoan": "Other / Non-rap / Uncategorized",
    "888 Blue": "Other / Non-rap / Uncategorized",
    "The infinite source": "Other / Non-rap / Uncategorized",
    "j. sula": "Other / Non-rap / Uncategorized",
    "Dr@k01387": "Other / Non-rap / Uncategorized",

    # US underground / independent / blog era
    "Oral Bee": "Underground / Independent Rap",
    "Joey Trap": "SoundCloud Rap",
    "Chris Webby": "Independent / Lyrical Rap",
    "XV": "Blog Era / Alternative Rap",
    "Caskey": "Independent / Street Rap",
    "King Los": "Technical / Lyrical Rap",
    "Dizzy Wright": "Independent / Conscious Rap",
    "Futuristic": "Independent / Pop Rap",
    "Smoke DZA": "Underground / Stoner Rap",
    "Iamsu!": "Bay Area / Mainstream Rap",
    "Philthy Rich": "Bay Area / Street Rap",
    "Slim Thug": "Southern Rap",
    "Rich Homie Quan": "Trap / Southern Rap",
    "Pi'erre Bourne": "Producer-Rapper / Trap",
    "Xanman": "Street Rap",
    "Young Igi": "Polish Rap",
    "Childish Gambino": "Alternative / Pop Rap",
    "BROCKHAMPTON": "Alternative Hip-Hop",
    "Lecrae": "Christian Rap",
    "Yelawolf": "Country Rap / Southern Rap",
    "Yzomandias": "Czech Rap",
    "Azure The Paradox": "Underground / Experimental Rap",
    "RUSSELL!": "Independent / Pop Rap",
    "Russell!": "Independent / Pop Rap",

    # Tyler duplicate / shortened name issue
    "Tyler": "Alternative Hip-Hop",
    "Tyler, The Creator": "Alternative Hip-Hop",

    # Trap / street / melodic additions
    "Lil Woodryc": "Street Rap",
    "Lil Tecca": "Melodic Rap",
    "Lancey Foux": "UK Rap / Experimental Trap",
    "Yung Beef": "Spanish Trap",
    "Yung Bans": "SoundCloud Rap",
    "A2H": "French Rap",
    "Chetta": "Dark Trap / Underground Rap",
    "Egreen": "Italian Rap",
    "Justin Stone": "Independent / Pop Rap",
    "DEATH PLUS": "Dark Trap / Underground Rap",
    "Sickboyrari": "Cloud Rap / Underground Rap",

    # Nerdcore / internet
    "Lee HendriX$on": "Nerdcore / YouTube Rap",
    "JT Music": "Nerdcore / YouTube Rap",
    "Dan Bull": "Nerdcore / YouTube Rap",
    "Rustage": "Nerdcore / YouTube Rap",
    "Akira The Don": "Spoken Word / Rap-adjacent",

    # International: French
    "Falcko": "French Rap",
    "Swift Guad": "French Rap",
    "LIM": "French Rap",
    "Gu": "French Rap",
    "Lucio Bukowski": "French Rap",
    "Ghetts": "UK Rap / Grime",
    "Vald": "French Rap",
    "Booba": "French Rap",
    "Rohff": "French Rap",
    "La Fouine": "French Rap",
    "Ninho": "French Rap",
    "Niro": "French Rap",
    "Lacrim": "French Rap",
    "Alkpote": "French Rap",
    "Disiz": "French Rap",
    "IAM": "French Rap",
    "MC Solaar": "French Rap",
    "Guizmo": "French Rap",
    "Kery James": "French Rap",

    # International: German / Austrian
    "Sentino": "German / Polish Rap",
    "Absztrakkt": "German Rap",
    "Cr7z": "German Rap",
    "Edo Saiya": "German Rap",
    "Prezident": "German Rap",
    "Fard": "German Rap",
    "Ali As": "German Rap",
    "257ers": "German Rap",
    "Azad": "German Rap",
    "RAF Camora": "German Rap",
    "Olexesh": "German Rap",
    "morten": "German Rap",
    "Chakuza": "German Rap",
    "Kollegah": "German Rap",
    "Bushido": "German Rap",
    "Fler": "German Rap",
    "Eko Fresh": "German Rap",
    "Money Boy": "German Rap",
    "Prinz Pi": "German Rap",
    "Samy Deluxe": "German Rap",
    "Kool Savas": "German Rap",

    # International: Italian
    "Mondo Marcio": "Italian Rap",
    "Fabri Fibra": "Italian Rap",
    "Gemitaiz": "Italian Rap",
    "Jesto": "Italian Rap",

    # International: Polish / Eastern Europe / Russian
    "Dudek P56": "Polish Rap",
    "Pikers": "Polish Rap",
    "Tede": "Polish Rap",
    "O.S.T.R.": "Polish Rap",
    "Peja": "Polish Rap",
    "Noize MC": "Russian Rap",
    "Johnyboy": "Russian Rap",
    "Kaisa": "German Rap",
    "Sagopa Kajmer": "Turkish Rap",
    "Sansar Salvo": "Turkish Rap",
    "MC Igu": "Brazilian Rap",
    "Blue Virus": "Italian Rap",
    "NANE": "Romanian Rap",
    "Swings ()": "Korean Rap",
    "101Barz": "Rap Platform / Freestyle",
    "O'hene Savant": "Underground / Independent Rap",

    # Dancehall / Latin / rap-adjacent
    "Vybz Kartel": "Dancehall / Rap-adjacent",
    "Daddy Yankee": "Reggaeton / Rap-adjacent",
    "Anuel AA": "Latin Trap / Reggaeton",

    # Possible aliases / malformed names from dataset
    "  (dima bamberg)": "Russian Rap",
    "  (vishel pokurit)": "Russian Rap",
    "  (Slava KPSS)": "Russian Rap",
    "  (naitivihod)": "Russian Rap",
    "  (Zamay)": "Russian Rap",
    "  (King SD)": "Russian Rap",
    "  (MEZZA)": "Russian Rap",
})

In [49]:
def broad_rap_category(cat):
    if cat is None:
        return "Other"

    c = cat.lower()

    if any(x in c for x in ["trap", "southern", "street", "gangsta", "drill"]):
        return "Trap / Street / Drill"

    if any(x in c for x in ["emo", "melodic", "cloud", "soundcloud", "plugg", "rage"]):
        return "Melodic / Emo / Cloud"

    if any(x in c for x in ["lyrical", "conscious", "technical", "boom bap"]):
        return "Lyrical / Conscious / Boom Bap"

    if any(x in c for x in ["classic", "old school", "golden age", "west coast", "east coast", "bay area"]):
        return "Classic / Regional Hip-Hop"

    if any(x in c for x in ["experimental", "abstract", "alternative", "underground", "independent"]):
        return "Alternative / Underground"

    if any(x in c for x in ["pop", "mainstream"]):
        return "Mainstream / Pop Rap"

    if any(x in c for x in ["horrorcore", "dark", "metal", "industrial"]):
        return "Dark / Horrorcore / Trap Metal"

    if any(x in c for x in ["uk", "grime"]):
        return "UK Rap / Grime"

    if any(x in c for x in ["french", "german", "polish", "italian", "russian", "turkish", "latin", "mexican", "brazilian", "korean", "romanian", "spanish", "czech"]):
        return "International Rap"

    if any(x in c for x in ["nerdcore", "youtube", "comedy", "battle"]):
        return "Internet / Battle / Comedy Rap"

    if any(x in c for x in ["r&b", "dancehall", "reggaeton", "musical theater", "spoken word", "rap-adjacent"]):
        return "Rap-adjacent"

    return "Other"

## Example Filters

In [34]:
import polars as pl
import re
import unicodedata

def normalize_artist_name(name):
    if name is None:
        return None

    name = str(name)

    # Replace HTML-ish spacing
    name = name.replace("&nbsp;", " ")

    # Remove accents safely
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))

    # Lowercase and normalize punctuation/spacing
    name = name.lower().strip()
    name = re.sub(r"\s+", " ", name)

    return name

artist_counts = (
    rap_df
    .group_by("artist")
    .agg(pl.len().alias("song_count"))
    .sort("song_count", descending=True)
)

pl.Config.set_tbl_rows(-1)

category_df = pl.DataFrame({
    "artist_norm": [normalize_artist_name(k) for k in artist_category_map.keys()],
    "rap_category": list(artist_category_map.values())
}).unique("artist_norm")

artist_counts_categorized = (
    artist_counts
    .with_columns(
        pl.col("artist")
        .map_elements(normalize_artist_name, return_dtype=pl.String)
        .alias("artist_norm")
    )
    .join(category_df, on="artist_norm", how="left")
    .with_columns(
        pl.when(pl.col("artist").str.contains("(?i)genius|translations|traducciones|tradues|romanizations|spotify"))
        .then(pl.lit("Other / Non-rap / Metadata"))
        .otherwise(pl.col("rap_category"))
        .alias("rap_category")
    )
    .with_columns(
        pl.col("rap_category").fill_null("Other / Non-rap / Uncategorized")
    )
    .drop("artist_norm")
    .sort(["rap_category", "song_count"], descending=[False, True])
)


category_summary = (
    artist_counts_categorized
    .group_by("rap_category")
    .agg([
        pl.len().alias("artist_count"),
        pl.col("song_count").sum().alias("total_songs")
    ])
    .sort("total_songs", descending=True)
)

still_uncategorized = (
    artist_counts_categorized
    .filter(pl.col("rap_category") == "Other / Non-rap / Uncategorized")
    .sort("song_count", descending=True)
)


Clean Artist Name

In [35]:
def clean_artist_name(name):
    if name is None:
        return None

    name = str(name)

    # Decode common HTML spacing
    name = name.replace("&nbsp;", " ")

    # Normalize accents
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))

    # Remove excessive whitespace
    name = re.sub(r"\s+", " ", name).strip()

    # Remove surrounding quotes if present
    name = name.strip('"').strip("'").strip()

    return name if name else None

In [37]:
rap_clean = (
    rap_df
    .with_columns(
        pl.col("artist")
        .map_elements(clean_artist_name, return_dtype=pl.String)
        .alias("artist_clean")
    )
)

metadata_pattern = (
    r"(?i)"
    r"genius|translations|traducciones|tradues|traductions|"
    r"romanizations|spotify|unknown artist|various artists"
)

rap_clean = (
    rap_clean
    .filter(~pl.col("artist_clean").str.contains(metadata_pattern))
)

In [38]:
artist_category_lookup = (
    artist_counts_categorized
    .select([
        pl.col("artist").map_elements(clean_artist_name, return_dtype=pl.String).alias("artist_clean"),
        "rap_category"
    ])
    .unique("artist_clean")
)

rap_clean = (
    rap_clean
    .join(artist_category_lookup, on="artist_clean", how="left")
)

In [39]:
bad_categories = [
    "Other / Non-rap / Metadata",
    "Other / Non-rap / Unknown",
    "Other / Non-rap / Compilation",
    "Other / Non-rap / Uncategorized",
    "Uncategorized",
]

rap_clean = (
    rap_clean
    .filter(~pl.col("rap_category").is_in(bad_categories))
    .filter(pl.col("rap_category").is_not_null())
)

Clean Lyrics

In [40]:
def clean_lyrics_text(text):
    if text is None:
        return None

    text = str(text)

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove bracketed section headers like [Chorus], [Verse 1], [Intro]
    text = re.sub(r"\[(intro|verse|chorus|hook|bridge|outro|pre-chorus|refrain|skit|interlude).*?\]", "", text, flags=re.I)

    # Remove common Genius embed/contributor junk if present
    text = re.sub(r"\d+ Contributors?.*?Lyrics", "", text, flags=re.I | re.S)
    text = re.sub(r"You might also like", "", text, flags=re.I)

    # Remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Normalize spacing inside lines
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.split("\n")]
    lines = [line for line in lines if line]

    text = "\n".join(lines).strip()

    return text if text else None

In [41]:
rap_clean = (
    rap_clean
    .with_columns(
        pl.col("lyrics")
        .map_elements(clean_lyrics_text, return_dtype=pl.String)
        .alias("lyrics_clean")
    )
    .filter(pl.col("lyrics_clean").is_not_null())
)

In [42]:
rap_clean = (
    rap_clean
    .with_columns([
        pl.col("lyrics_clean").str.len_chars().alias("lyrics_chars"),
        pl.col("lyrics_clean").str.split("\n").list.len().alias("line_count"),
    ])
    .filter(pl.col("lyrics_chars") >= 500)
    .filter(pl.col("lyrics_chars") <= 12000)
    .filter(pl.col("line_count") >= 8)
)

In [44]:
rap_clean = (
    rap_clean
    .unique(subset=["artist_clean", "title"], keep="first")
)

rap_clean = (
    rap_clean
    .unique(subset=["lyrics_clean"], keep="first")
)

In [46]:
rap_english = (
    rap_clean
    .filter(
        (
            pl.col("language")
            .cast(pl.String)
            .str.to_lowercase()
            .is_in(["en", "eng", "english"])
        )
        |
        (
            pl.col("language_cld3")
            .cast(pl.String)
            .str.to_lowercase()
            .is_in(["en", "eng", "english"])
        )
        |
        (
            pl.col("language_ft")
            .cast(pl.String)
            .str.to_lowercase()
            .is_in(["en", "eng", "english"])
        )
    )
)

rap_english.shape

(81392, 16)

In [47]:
rap_english.write_parquet("rap_english_clean_categorized.parquet")
rap_english.write_csv("rap_english_clean_categorized.csv")

In [50]:
rap_english = (
    rap_english
    .with_columns(
        pl.col("rap_category")
        .map_elements(broad_rap_category, return_dtype=pl.String)
        .alias("rap_family")
    )
)

In [51]:
family_summary = (
    rap_english
    .group_by("rap_family")
    .agg([
        pl.len().alias("song_count"),
        pl.col("artist_clean").n_unique().alias("artist_count"),
        pl.col("rap_category").n_unique().alias("source_categories"),
        pl.col("lyrics_chars").mean().alias("avg_chars"),
        pl.col("line_count").mean().alias("avg_lines"),
    ])
    .sort("song_count", descending=True)
)

family_summary

rap_family,song_count,artist_count,source_categories,avg_chars,avg_lines
str,u32,u32,u32,f64,f64
"""Trap / Street / Drill""",25964,97,28,2535.167424,60.237059
"""Melodic / Emo / Cloud""",13732,49,15,1983.306438,49.341028
"""Lyrical / Conscious / Boom Bap""",12501,46,16,2868.462363,66.513319
"""Classic / Regional Hip-Hop""",8367,36,6,2949.582168,72.26975
"""Alternative / Underground""",6725,34,20,2265.35658,55.813234
"""Mainstream / Pop Rap""",5995,21,5,2528.676731,63.786322
"""Internet / Battle / Comedy Rap""",2640,17,5,3049.185606,73.534091
"""UK Rap / Grime""",1829,10,2,2894.185894,74.08912
"""Dark / Horrorcore / Trap Metal""",1793,6,2,2891.88232,69.068042


In [52]:
rap_english.write_parquet("rap_english_clean_categorized_with_families.parquet")
rap_english.write_csv("rap_english_clean_categorized_with_families.csv")